In [30]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape, Polygon
import numpy as np
from pathlib import Path
import os
import sys

In [3]:
CURRENT_DIRECTORY =  os.getcwd()
PARENT_DIRECTORY = os.path.dirname(CURRENT_DIRECTORY)
print(PARENT_DIRECTORY)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src


In [4]:
BASE_DATASET_PATH = Path(PARENT_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [16]:
# QGIS_PATH = '/Applications/QGIS-LTR.app/Contents/Resources/python' 

# # Add the QGIS Python path to the system path
# sys.path.append(QGIS_PATH)
# # Add path to plugins directory (usually needed for 'processing' module)
# sys.path.append(os.path.join(QGIS_PATH, 'plugins'))

# # QgsApplication.setPrefixPath() tells QGIS where to look for its core C++ libraries (plugins, resources, etc.).
# # The first argument is the QGIS installation prefix (usually the folder containing 'bin', 'apps', etc.)
# # For your Mac example, we use the path leading up to 'Resources'.
# qgis_app_prefix = '/Applications/QGIS-LTR.app/Contents/Resources' # Adjust as needed

# print(f"1. Setting QGIS prefix path to: {qgis_app_prefix}")
# from qgis.core import QgsApplication
# QgsApplication.setPrefixPath(qgis_app_prefix, True)

# # Create an instance of QgsApplication. The second argument (False) means no GUI.
# # This prepares the environment for non-GUI (standalone) scripting.
# qgs = QgsApplication([], False) 

# # Load Providers and registries (REQUIRED)
# print("2. Initializing QGIS application...")
# qgs.initQgis() 
# print("QGIS application initialized successfully.")

In [17]:
# # Import QGIS Libraries and Processing
# try:
#     # Now that QGIS is initialized, we can safely import these modules
#     from qgis.analysis import QgsNativeAlgorithms
#     import processing
#     from processing.core.Processing import Processing
#     from qgis.core import QgsProject, QgsProcessingFeedback, QgsVectorLayer, QgsCoordinateReferenceSystem
#     from qgis.PyQt.QtCore import QVariant 
# except ImportError as e:
#     print(f"\n--- FATAL ERROR: MODULE IMPORT FAILED AFTER INITIALIZATION ---")
#     print(f"Please double-check the QGIS_PATH and qgis_app_prefix variables.")
#     print(f"Original error: {e}")
#     sys.exit(1)

In [18]:
### Uncomment to run qgis related commands
# # initialise the Processing Framework
# Processing.initialize()
# # Add the QGIS native algorithms provider
# QgsApplication.processingRegistry().addProvider(QgsNativeAlgorithms())

In [19]:
# reading singapore 2021 multipolygon map data
singapore_2021_filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_2021_multipolygons.gpkg"
singapore_2021_gpd = gpd.read_file(singapore_2021_filepath)
singapore_2021_gpd.crs

<Projected CRS: EPSG:3414>
Name: SVY21 / Singapore TM
Axis Info [cartesian]:
- N[north]: Northing (metre)
- E[east]: Easting (metre)
Area of Use:
- name: Singapore - onshore and offshore.
- bounds: (103.59, 1.13, 104.07, 1.47)
Coordinate Operation:
- name: Singapore Transverse Mercator
- method: Transverse Mercator
Datum: SVY21
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [20]:
# add a manual_tags column
singapore_2021_gpd["manual_tags"] = None
singapore_2021_gpd.columns

Index(['osm_id', 'osm_way_id', 'name', 'type', 'aeroway', 'amenity',
       'admin_level', 'barrier', 'boundary', 'building', 'craft', 'geological',
       'historic', 'land_area', 'landuse', 'leisure', 'man_made', 'military',
       'natural', 'office', 'place', 'shop', 'sport', 'tourism', 'other_tags',
       'geometry', 'manual_tags'],
      dtype='object')

In [21]:
# commercial column is not descriptive enough. 
# rows where commercial != null AND other_tags contains HDB will be tagged with street_shops
# rows where commerical != nulul AND shop == "mall" will be tagged with mall
singapore_landuse = singapore_2021_gpd[singapore_2021_gpd["landuse"] == "commercial"].copy()

hdb_condition = singapore_landuse["other_tags"].str.contains("HDB", na = False)
mall_condition = singapore_landuse["shop"] == "mall"

singapore_landuse[["other_tags", "shop"]] = singapore_landuse[["other_tags", "shop"]].astype(str)

singapore_landuse.loc[hdb_condition, "manual_tags"] = "street_shops"
singapore_landuse.loc[mall_condition, "manual_tags"] = "mall"

singapore_2021_gpd.update(singapore_landuse[["manual_tags"]])

file_path = str(BASE_DATASET_PATH / "geospatial_data/2021_geospatial/commercial_polygons.xlsx")
singapore_landuse.to_excel(file_path)

singapore_2021_gpd["manual_tags"].unique()

array([None, 'mall', 'street_shops'], dtype=object)

#### Removing the Former Tanjong Pagar Railway Station for column building = train_station

In [22]:
before = singapore_2021_gpd.shape[0]
singapore_2021_gpd = singapore_2021_gpd[singapore_2021_gpd["osm_way_id"] != "393439098"]
after = singapore_2021_gpd.shape[0]

print(before, after)

125476 125475


In [23]:
# collating the changes and saving to another gpkg file
filepath = str(BASE_DATASET_PATH / "geospatial_data/2021_geospatial/multipolygons_edited_2021.gpkg")
singapore_2021_gpd.to_file(filepath)

In [11]:
singapore_2021_lines_filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_2021_lines.gpkg"
singapore_2021_lines = gpd.read_file(singapore_2021_lines_filepath)
singapore_2021_lines.crs

<Projected CRS: EPSG:3414>
Name: SVY21 / Singapore TM
Axis Info [cartesian]:
- N[north]: Northing (metre)
- E[east]: Easting (metre)
Area of Use:
- name: Singapore - onshore and offshore.
- bounds: (103.59, 1.13, 104.07, 1.47)
Coordinate Operation:
- name: Singapore Transverse Mercator
- method: Transverse Mercator
Datum: SVY21
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [12]:
# https://wiki.openstreetmap.org/wiki/WikiProject_Singapore#Roads
print(f"{singapore_2021_lines.columns} \n")
print(singapore_2021_lines['highway'].unique())

Index(['osm_id', 'name', 'highway', 'waterway', 'aerialway', 'barrier',
       'man_made', 'z_order', 'other_tags', 'geometry'],
      dtype='object') 

['primary' 'residential' 'tertiary' 'footway' 'service' 'secondary'
 'motorway' 'motorway_link' 'trunk' None 'trunk_link' 'primary_link'
 'secondary_link' 'living_street' 'construction' 'pedestrian'
 'unclassified' 'proposed' 'steps' 'track' 'cycleway' 'path'
 'tertiary_link' 'bridleway' 'raceway' 'corridor' 'elevator' 'rest_area']


#### Based on the wiki, I have decided to include motorway, trunk, primary, secondary as the major roads in singapore to include

In [13]:
major_roads = [
    'motorway', 
    'trunk', 
    'primary', 
    'secondary'
]

singapore_major_roads = singapore_2021_lines[
    singapore_2021_lines['highway'].isin(major_roads)
].copy()

file_path = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_major_roads_2021.gpkg"
singapore_major_roads.to_file(file_path)

#### Combine the lines of major roads into one singular line
There will be feature loss but the aim was to create a polygon of the major roads. There wasnt a need to preserve the road's features.

In [14]:
combined_major_roads = singapore_major_roads.dissolve()
combined_major_roads.columns
# singapore_2021_gpd.columns

Index(['geometry', 'osm_id', 'name', 'highway', 'waterway', 'aerialway',
       'barrier', 'man_made', 'z_order', 'other_tags'],
      dtype='object')

In [15]:
drop_columns = ["osm_id", "name", "highway", "waterway", "aerialway", "barrier", "man_made", "z_order", "other_tags"]
combined_major_roads = combined_major_roads.drop(drop_columns, axis = 1, errors = 'ignore')
combined_major_roads["major_roads"] = "yes"
print(combined_major_roads)
file_path = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_combined_major_roads_2021.gpkg"
# # save to gpkg file for processing using qgis
combined_major_roads.to_file(file_path)

                                            geometry major_roads
0  MULTILINESTRING ((27637.517 32038.397, 27643.1...         yes


In [ ]:
# transforming the lines into polygon using the buffer function in qgis
processing.run("native:buffer", {'INPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/singapore_combined_major_roads_2021.gpkg|layername=singapore_combined_major_roads_2021','DISTANCE':5,'SEGMENTS':5,'END_CAP_STYLE':0,'JOIN_STYLE':0,'MITER_LIMIT':2,'DISSOLVE':True,'SEPARATE_DISJOINT':False,'OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/2021_combined_roads_polygon.gpkg'})

### Creating a grid over Singapore using qgis
The grid size will be a hectare large ($100m \times 100m$)

The extents of the grid was determined by looking at Google maps to find the boundaries of Singapore. And converting to 

In [18]:
processing.run("native:creategrid", {'TYPE':2,'EXTENT':'2666.773500000,56466.773500000,15657.950600000,50257.950600000 [EPSG:3414]','HSPACING':100,'VSPACING':100,'HOVERLAY':0,'VOVERLAY':0,'CRS':QgsCoordinateReferenceSystem('EPSG:3414'),'OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/hectare_grid.gpkg'})

{'OUTPUT': '/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/hectare_grid.gpkg'}

```
ogr2ogr -f PostgreSQL \
  PG:"dbname=postgis user=yitong" \
  hectare_grid.gpkg \
  -nln hectare_grid \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  hectare_grid

ogr2ogr -f PostgreSQL \
  PG:"dbname=postgis user=yitong" \
  2021_combined_roads_polygon.gpkg \
  -nln 2021_combined_roads_polygon \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  2021_combined_roads_polygon

ogr2ogr -f PostgreSQL \
  PG:"dbname=postgis user=yitong" \
  multipolygons_edited_2021.gpkg \
  -nln multipolygons_edited_2021 \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  multipolygons_edited_2021
```
#### Fixing invalid geometries
```
UPDATE multipolygons_edited_2021
SET geom = ST_CollectionExtract(ST_MakeValid(geom), 3)::geometry(MultiPolygon, 3414)
WHERE NOT ST_IsValid(geom);
```
#### Intersecting multipolygons with hectare grid
```
DROP TABLE IF EXISTS polygons_grid_2021;

CREATE TABLE polygons_grid_2021 AS
SELECT 
    g.id AS grid_id,
    g.row_index as grid_row,
    g.col_index as grid_col,
    mp.id AS id,
    mp.osm_id as osm_id,
    mp.osm_way_id as osm_way_id,
    mp.name AS name,
	mp.amenity as amenity,
	mp.building as building,
	mp.landuse as landuse,
	mp.military as military,
	mp.office as office,
	mp.place as place,
	mp.shop as shop,
	mp.other_tags as other_tags,
  mp.manual_tags as manual_tags,
    ST_Intersection(ST_MakeValid(mp.geom), g.geom) AS geom
FROM hectare_grid g
JOIN multipolygons_edited_2021 mp
  ON ST_Intersects(g.geom, mp.geom);
```
#### Intersecting road polygons with hectare grid
```
DROP TABLE IF EXISTS roads_grid_2021;

Create TABLE roads_grid_2021 AS
SELECT
  g.id AS grid_id,
  g.row_index as grid_row,
  g.col_index as grid_col,
  mp.major_roads as major_roads,
  ST_Intersection(ST_MakeValid(mp.geom), g.geom) AS geom
  FROM hectare_grid g
  JOIN "2021_combined_roads_polygon" mp
  ON ST_Intersects(g.geom, mp.geom);
```

#### Combining the 2 tables
```
DROP TABLE IF EXISTS combined_grid_2021;
CREATE TABLE combined_grid_2021 AS
SELECT 
    grid_id,
    grid_row,
    grid_col,
    geom,
    major_roads,
    NULL::integer AS id,
    NULL::varchar AS osm_way_id,
    NULL::varchar AS name,
    NULL::varchar AS amenity,
    NULL::varchar AS building,
    NULL::varchar AS landuse,
    NULL::varchar AS military,
    NULL::varchar AS office,
    NULL::varchar AS place,
    NULL::varchar AS shop,
    NULL::varchar AS other_tags,
    NULL::varchar AS manual_tags,
    NULL::varchar AS osm_id
FROM roads_grid_2021

UNION ALL

SELECT 
    grid_id,
    grid_row,
    grid_col,
    geom,
    NULL::text AS major_roads,
    id,
    osm_way_id,
    name,
    amenity,
    building,
    landuse,
    military,
    office,
    place,
    shop,
    other_tags,
    manual_tags,
    osm_id
FROM polygons_grid_2021;
```

The below command shows that the newly created table does not have a primary key
```
SELECT
    kcu.column_name
FROM
    information_schema.table_constraints AS tc
JOIN
    information_schema.key_column_usage AS kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
WHERE
    tc.table_name = 'polygons_grid_2021'
    AND tc.table_schema = 'public'
    AND tc.constraint_type = 'PRIMARY KEY';
```
#### creating a primary key with uid as the column name
```
ALTER TABLE polygons_grid_2021
ADD COLUMN uid SERIAL PRIMARY KEY;
```



In [ ]:
filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/polygon_grids_testing.gpkg"
combined_2021 = gpd.read_file(filepath)
combined_2021

,grid_id,grid_row,grid_col,major_roads,id,osm_way_id,name,amenity,building,landuse,military,office,place,shop,other_tags,manual_tags,osm_id,geometry
0,3345,230,9,yes,NaN,None,None,None,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((3616.49 27257.951, 3626.49 272..."
1,3346,231,9,yes,NaN,None,None,None,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((3616.089 27157.951, 3626.089 2..."
2,3340,225,9,yes,NaN,None,None,None,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((3649.955 27683.196, 3650.068 2..."
3,3683,222,10,yes,NaN,None,None,None,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((3759.744 27990.092, 3766.774 2..."
4,3347,232,9,yes,NaN,None,None,None,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((3615.434 26994.786, 3615.687 2..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887675,186008,205,537,None,366.0,None,Southeast,None,None,None,None,None,None,None,"""name:zh""=>""东南区"",""wikidata""=>""Q1687545"",""ISO31...",None,3831715,"POLYGON ((56466.774 29757.951, 56466.774 29657..."
887676,186009,206,537,None,1.0,None,Singapore,None,None,None,None,None,None,None,"""flag""=>""http://upload.wikimedia.org/wikipedia...",None,536780,"POLYGON ((56466.774 29657.951, 56466.774 29557..."
887677,186009,206,537,None,366.0,None,Southeast,None,None,None,None,None,None,None,"""name:zh""=>""东南区"",""wikidata""=>""Q1687545"",""ISO31...",None,3831715,"POLYGON ((56466.774 29657.951, 56466.774 29557..."
887678,186010,207,537,None,1.0,None,Singapore,None,None,None,None,None,None,None,"""flag""=>""http://upload.wikimedia.org/wikipedia...",None,536780,"POLYGON ((56466.774 29557.951, 56466.774 29457..."


In [50]:
shorten_combined_2021 = combined_2021[
    (combined_2021["grid_id"] <= 39274) & 
    (combined_2021["grid_id"] > 39273)
].sort_values(by="grid_id")
shorten_combined_2021.shape

(10, 18)

In [ ]:
# to keep track of the polygon areas of each hectare
hectare_objects = {"airport": 0, "mall": 0, "schools": 0, "public_transport_station": 0,
             "major_road": 0, "residential_area": 0, "workforce": 0}

CHANGI_AIRPORT = '174037464'
SELETAR_AIRPORT = '381760619'
AIRPORT = "airport"

In [25]:
print(shorten_combined_2021.columns)
shorten_combined_2021.head(2)

Index(['grid_id', 'grid_row', 'grid_col', 'major_roads', 'id', 'osm_way_id',
       'name', 'amenity', 'building', 'landuse', 'military', 'office', 'place',
       'shop', 'other_tags', 'manual_tags', 'osm_id', 'geometry'],
      dtype='object')


,grid_id,grid_row,grid_col,major_roads,id,osm_way_id,name,amenity,building,landuse,military,office,place,shop,other_tags,manual_tags,osm_id,geometry
60293,19996,273,57,None,367.0,None,Southwest,None,None,None,None,None,None,None,"""name:zh""=>""西南区"",""wikidata""=>""Q5784126"",""ISO31...",None,3831716,"POLYGON ((8466.774 22957.951, 8466.774 22857.9..."
60294,19996,273,57,None,119658.0,795855452,Western Islands,None,None,None,None,None,suburb,None,"""name:en""=>""Western Islands"",""name:ms""=>""Kepul...",None,None,"POLYGON ((8466.774 22957.951, 8466.774 22857.9..."


[0'grid_id', 1'grid_row', 2'grid_col', 3'major_roads', 4'id', 5'osm_way_id',
       6'name', 7'amenity', 8'building', 9'landuse', 10'military', 11'office', 12'place',
       13'shop', 14'other_tags', 15'manual_tags', 16'osm_id', 17'geometry']

In [ ]:
def classify_hectare(input_pd):

    current_grid_id = 0
    temp_dict = hectare_objects

    for r in input_pd.itertuples(index=False):
        # if the hectare contains airport polygons
        if r[5] == CHANGI_AIRPORT or r[5] == SELETAR_AIRPORT:
            temp_dict = calculate_area(r, temp_dict, AIRPORT)
            # update dataframe

        # if the grid_id incremented, means we moving onto the next hectare grid
        if current_grid_id != r[0] and current_grid_id != 0:
            classification = decide_classification(current_grid_id, temp_dict)
            # update dataframe

        current_grid_id = r[0]

        
        print(r)

def calculate_area(row, temp_dict, classification):
    if classification == AIRPORT:
        # if the hectare contains airport, that hectare will be classified as airport
        # so make the airport area be the largest
        temp_dict["airport"] += 10000
        return temp_dict
    
    # if not airport or mall, add up the area of that classification
    temp_dict[classification] += row[17].area
    return temp_dict

def decide_classification(current_grid_id, temp_dict):

    

In [ ]:
classify_hectare(shorten_combined_2021)

In [58]:
for r in shorten_combined_2021.itertuples(index=False):
    print(r)
for r in shorten_combined_2021.itertuples(index=False):
    print(r[17].area)

Pandas(grid_id=39274, grid_row=175, grid_col=113, major_roads=None, id=9257.0, osm_way_id='117920012', name='9', amenity=None, building='yes', landuse=None, military=None, office=None, place=None, shop=None, other_tags=None, manual_tags=None, osm_id=None, geometry=<POLYGON ((14066.774 32682.745, 14066.774 32657.951, 13993.893 32657.951, 13...>)
Pandas(grid_id=39274, grid_row=175, grid_col=113, major_roads=None, id=73964.0, osm_way_id='453244612', name=None, amenity=None, building='yes', landuse=None, military=None, office=None, place=None, shop=None, other_tags=None, manual_tags=None, osm_id=None, geometry=<POLYGON ((14046.958 32746.161, 14046.879 32736.928, 14020.715 32737.161, 14...>)
Pandas(grid_id=39274, grid_row=175, grid_col=113, major_roads=None, id=73963.0, osm_way_id='453244611', name=None, amenity=None, building='yes', landuse=None, military=None, office=None, place=None, shop=None, other_tags=None, manual_tags=None, osm_id=None, geometry=<POLYGON ((14046.98 32748.682, 14047.